In [1]:
import pandas as pd
import openpyxl
from api_data_clients import get_entreprise_infos

In [15]:
df_clients = pd.read_excel('Base de données Clients.xlsx')

In [16]:
df_clients.shape

(81, 13)

## df_clients



In [17]:
import re

df_clients['code_postal'] = df_clients['Adresse de Facturation 4'].str.extract(r'(\d{5})')

In [23]:
# Version qui collecte aussi les erreurs ou absences de résultats

resultats = []
erreurs = []

for idx, row in df_clients.iterrows():
    nom = str(row['Nom de l’entité légale à facturer'])
    code_postal = str(row['code_postal'])
    adresse = str(row['Adresse de Facturation 1'])
    try:
        df = get_entreprise_infos(nom, adresse=adresse)
        if df.empty:
            erreurs.append({"Nom": nom, "Code Postal": code_postal, "Adresse": adresse, "Erreur": "Aucun résultat"})
        else:
            resultats.append(df)
    except Exception as e:
        erreurs.append({"Nom": nom, "Code Postal": code_postal, "Adresse": adresse, "Erreur": str(e)})

# Concatène les résultats valides
if resultats:
    df_resultats = pd.concat(resultats, ignore_index=True)
else:
    df_resultats = pd.DataFrame()

# Affiche les résultats valides
print("Résultats valides :")
display(df_resultats)

# Affiche les erreurs ou absences de résultats
if erreurs:
    df_erreurs = pd.DataFrame(erreurs)
    print("Erreurs ou absences de résultats :")
    display(df_erreurs)
else:
    print("Aucune erreur.")

Résultats valides :


,Nom Entreprise,Adresse,Code Postal,SIREN,TVA
0,2050 NOW LA MAISON,10 BOULEVARD DE GRENELLE 75015 PARIS,75015,447906165,FR11447906165
1,2050 NOW LE MEDIA,10 BOULEVARD DE GRENELLE 75015 PARIS,75015,922468772,FR14922468772
2,ACCOR (ACCOR),82 RUE H FARMAN 92130 ISSY LES MOULINEAUX,92130,602036444,FR93602036444
3,ACPM (ALPES CUIR & PEAUX DU MIDI),931 ROUTE VIEILLE 30260 LIOUC,30260,849232129,FR32849232129
4,AFRICA MEDIA GROUP,57 B RUE D'AUTEUIL 75016 PARIS,75016,892312661,FR89892312661
...,...,...,...,...,...
69,UNION DES MARQUES (UDA),ETAGE 4 CHEZ MORNING 128 RUE DU FAUBOURG SAIN...,75008,784669533,FR01784669533
70,L'ECOLE VAN CLEEF & ARPELS (L ECOLE DES ARTS J...,22 PLACE VENDOME 75001 PARIS,75001,433901485,FR66433901485
71,VILLAGES CLUBS DU SOLEIL,23 RUE FRANCOIS SIMON 13003 MARSEILLE,13003,422636472,FR21422636472
72,WMH PROJECT (FC2 EVENTS),TOUR ALTO 4 PLACE DES SAISONS 92400 COURBEVOIE,92400,402423420,FR27402423420


Erreurs ou absences de résultats :


,Nom,Code Postal,Adresse,Erreur
0,Balenciaga S.A.S.,nan,40 Rue de Sèvres,Aucun résultat
1,BrutX,75002,51 rue Vivienne,Aucun résultat
2,CHANEL S.A.S GDP In-Store & Mob,92200,135 avenue Charles de Gaulle,Aucun résultat
3,HAPPENINGCO SAS,75015,10 Boulevard de Grenelle,Aucun résultat
4,LACOSTE OPERATIONS HQ,10080,"321, rue du Faubourg Croncels",Aucun résultat
5,LACOSTE OPERATIONS Market,75016,31-37 Bld de Montmorency,Aucun résultat
6,MOËT HENNESSY L.VUITTON,75008,22 avenue Montaigne,Aucun résultat


In [ ]:
df_resultats.to_excel('infos_clients_siren_TVA.xlsx', index=False)

In [18]:
def get_entreprise_infos(nom, code_postal=None, adresse=None):
    """
    Recherche une entreprise par nom (obligatoire), code postal (optionnel) et adresse (optionnelle).
    Retourne un DataFrame pandas avec Nom Entreprise, Adresse, Code Postal, SIREN, TVA.
    Prend uniquement le premier résultat retourné par l'API.
    """
    import requests
    import pandas as pd

    params = {
        "q": nom,
        "per_page": 5
    }
    if code_postal:
        params["code_postal"] = code_postal
    if adresse:
        params["adresse"] = adresse
    url = "https://recherche-entreprises.api.gouv.fr/search"
    headers = {"accept": "application/json", "User-Agent": "outil_devis_demo/1.0"}
    response = requests.get(url, params=params, headers=headers)
    if not response.ok:
        raise Exception(f"Erreur API: {response.status_code} {response.text}")
    data = response.json()
    results = data.get("results", [])
    if not results:
        return pd.DataFrame(columns=["Nom Entreprise", "Adresse", "Code Postal", "SIREN", "TVA"])
    r = results[0]
    nom_ent = r.get("nom_complet")
    siege = r.get("siege", {})
    adresse_ent = siege.get("adresse")
    code_postal_ent = siege.get("code_postal")
    siren = r.get("siren")
    def tva_fr(siren):
        siren = str(siren)
        if len(siren) != 9 or not siren.isdigit():
            return None
        cle = (12 + 3 * (int(siren) % 97)) % 97
        return f"FR{cle:02d}{siren}"
    tva = tva_fr(siren)
    row = {
        "Nom Entreprise": nom_ent,
        "Adresse": adresse_ent,
        "Code Postal": code_postal_ent,
        "SIREN": siren,
        "TVA": tva
    }
    return pd.DataFrame([row])


## app streamlit

In [38]:
df_entreprise = get_entreprise_infos('demain', '75002') 

df_entreprise

,Nom Entreprise,Adresse,Code Postal,SIREN,TVA
0,DEMAIN,2-4 2 RUE DE PALESTRO 75002 PARIS,75002,821317856,FR29821317856


2026-02-13 11:00:02.492 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-13 11:00:02.576 
  command:

    streamlit run C:\Users\Clément\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-13 11:00:02.577 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-13 11:00:02.577 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-13 11:00:02.578 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-13 11:00:02.578 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-13 11:00:02.578 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-13 11:00:02.578 Th